# Interview Preparation - KnowBe4

## Create fake data set

In [147]:
import pandas as pd
import numpy as np
from faker import Faker
from datetime import datetime, timedelta
import random


# Faker helps generate realistic fake data
fake = Faker()
# Set seeds for reproducibility
np.random.seed(42)
random.seed(42)


# Number of users
n_users = 500
# Create user IDs
user_ids = np.arange(1, n_users + 1)
# Possible countries
countries = [
    "Netherlands",
    "Germany",
    "South Africa",
    "USA",
    "UK",
    "France"
]
# Generate user dataset
users = pd.DataFrame({
    "user_id": user_ids,
    # Random country assignment
    "country": np.random.choice(countries, n_users),
    # Simulate account age in days
    "account_age_days": np.random.randint(1, 3000, n_users)
})
# Display first few rows
users.head()

# Number of events
n_events = 20000
# Event types
event_types = [
    "login",
    "logout",
    "file_download",
    "password_reset",
    "email_click",
    "vpn_access"
]
# Create empty list to store events
event_rows = []
# Generate event rows
for _ in range(n_events):
    # Random user
    user_id = np.random.choice(user_ids)
    # Random timestamp within last 30 days
    timestamp = (
        datetime.now() - timedelta(
            days=np.random.randint(0, 30),
            hours=np.random.randint(0, 24),
            minutes=np.random.randint(0, 60)
        )
    )
    # Random event type
    event_type = np.random.choice(event_types)
    # Generate fake IP address
    ip_address = fake.ipv4_public()
    # Simulate highly imbalanced target
    # About 2% threats
    is_threat = np.random.choice(
        [0, 1],
        p=[0.98, 0.02]
    )
    # Add suspicious behavior patterns
    # Threats more likely to happen at night
    if is_threat == 1:
        timestamp = timestamp.replace(
            hour=np.random.choice([1, 2, 3, 4])
        )
    # Store row
    event_rows.append([
        user_id,
        timestamp,
        event_type,
        ip_address,
        is_threat
    ])
# Create events DataFrame
events = pd.DataFrame(
    event_rows,
    columns=[
        "user_id",
        "timestamp",
        "event_type",
        "ip_address",
        "is_threat"
    ]
)
# Display first rows
events.head()

# Introduce some missing user IDs
missing_indices = np.random.choice(events.index, 100)
events.loc[missing_indices, "user_id"] = np.nan
# Introduce missing IP addresses
missing_ip_indices = np.random.choice(events.index, 50)
events.loc[missing_ip_indices, "ip_address"] = np.nan

# Save datasets
users.to_parquet("users.parquet", index=False)
events.to_parquet("events.parquet", index=False)
print("Parquet files successfully created!")

Parquet files successfully created!


## Load data

In [148]:
# Create two separate pandas dataframes
events = pd.read_parquet('events.parquet')
users = pd.read_parquet('users.parquet')

## Inspect data

In [149]:
events.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20000 entries, 0 to 19999
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype         
---  ------      --------------  -----         
 0   user_id     19902 non-null  float64       
 1   timestamp   20000 non-null  datetime64[ns]
 2   event_type  20000 non-null  object        
 3   ip_address  19950 non-null  object        
 4   is_threat   20000 non-null  int64         
dtypes: datetime64[ns](1), float64(1), int64(1), object(2)
memory usage: 781.4+ KB


In [150]:
users.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 500 entries, 0 to 499
Data columns (total 3 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   user_id           500 non-null    int64 
 1   country           500 non-null    object
 2   account_age_days  500 non-null    int64 
dtypes: int64(2), object(1)
memory usage: 11.8+ KB


## Join datasets

In [151]:
# Join events and users dataframes into a single df
df = pd.merge(
    events,
    users,
    on='user_id',
    how='left'
)
df

,user_id,timestamp,event_type,ip_address,is_threat,country,account_age_days
0,369.0,2026-04-09 12:32:40.976378,file_download,139.243.92.111,0,Netherlands,2682.0
1,194.0,2026-04-25 02:02:40.978525,logout,39.58.205.41,0,UK,1251.0
2,300.0,2026-05-05 06:44:40.979827,vpn_access,205.39.117.1,0,South Africa,268.0
3,212.0,2026-05-08 16:21:40.983088,password_reset,201.214.23.232,0,Netherlands,1032.0
4,90.0,2026-04-26 06:33:40.983178,vpn_access,189.146.183.16,0,UK,76.0
...,...,...,...,...,...,...,...
19995,480.0,2026-04-23 21:49:41.403221,password_reset,24.32.85.17,0,USA,2398.0
19996,277.0,2026-04-16 09:32:41.403238,file_download,117.82.159.45,0,Netherlands,714.0
19997,22.0,2026-04-14 22:28:41.403261,password_reset,168.59.60.229,0,Netherlands,2208.0
19998,100.0,2026-04-13 14:41:41.403280,vpn_access,217.230.162.156,0,UK,396.0


In [152]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20000 entries, 0 to 19999
Data columns (total 7 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   user_id           19902 non-null  float64       
 1   timestamp         20000 non-null  datetime64[ns]
 2   event_type        20000 non-null  object        
 3   ip_address        19950 non-null  object        
 4   is_threat         20000 non-null  int64         
 5   country           19902 non-null  object        
 6   account_age_days  19902 non-null  float64       
dtypes: datetime64[ns](1), float64(2), int64(1), object(3)
memory usage: 1.1+ MB


## Cleaning of data

In [153]:
# Fill nulls/NaN
df = df.fillna(0)

# User_id needs to be int64 type
df['user_id'] = df['user_id'].astype('int64')

# timestamp needs to be datetime type
df['timestamp'] = pd.to_datetime(df['timestamp'])

# make sure ip address is of object type
df['ip_address'] = df['ip_address'].astype('string')

In [154]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20000 entries, 0 to 19999
Data columns (total 7 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   user_id           20000 non-null  int64         
 1   timestamp         20000 non-null  datetime64[ns]
 2   event_type        20000 non-null  object        
 3   ip_address        20000 non-null  string        
 4   is_threat         20000 non-null  int64         
 5   country           20000 non-null  object        
 6   account_age_days  20000 non-null  float64       
dtypes: datetime64[ns](1), float64(1), int64(2), object(2), string(1)
memory usage: 1.1+ MB


## Feature Engineering
### This might be the most crucial part of the project, to detect behavioral signals
- Rolling window to check past activity of user
- index need to be changed to timestamp
- Add features like events per user for the last 24hrs, 12hrs, 1hr
- Add features ip_address related like new_ip_user, user_ip_addresses, ip_addresses_24hr

In [155]:
# Change index to timestamp for rolling window purposes
df = df.set_index('timestamp')

# Sort values for dataset to ensure rolling window could be used
df = df.sort_values(['user_id', 'timestamp'])
df

,user_id,event_type,ip_address,is_threat,country,account_age_days
timestamp,,,,,,
2026-04-09 19:06:41.072217,0,password_reset,39.10.96.188,0,0,0.0
2026-04-09 22:34:41.099105,0,file_download,183.176.206.92,0,0,0.0
2026-04-10 00:12:41.372096,0,email_click,81.167.201.129,0,0,0.0
2026-04-10 07:15:41.032578,0,file_download,188.199.224.205,0,0,0.0
2026-04-10 23:15:41.215272,0,email_click,33.196.189.113,0,0,0.0
...,...,...,...,...,...,...
2026-05-05 14:37:41.079960,500,login,66.45.57.92,0,USA,118.0
2026-05-05 22:37:41.366141,500,password_reset,69.163.139.75,0,USA,118.0
2026-05-07 04:07:41.230185,500,password_reset,180.16.246.9,0,USA,118.0


In [ ]:
# Create features for time related events per user
df['u1hr_events'] = (
    df.groupby('user_id')['user_id']
      .rolling('1h')
      .count()
      .groupby(level=0)
      .shift(1)
      .reset_index(level=0, drop=True)
      .fillna(0)
)

df['u12hr_events'] = (
    df.groupby("user_id")['user_id']           
      .rolling('12h')
      .count()
      .groupby(level=0)
      .shift(1)
      .reset_index(level=0, drop=True)
      .fillna(0)
)

df['u24hr_events'] = (
    df.groupby('user_id')['user_id']
      .rolling('24h')
      .count()
      .groupby(level=0)
      .shift(1)
      .reset_index(level=0, drop=True)
      .fillna(0)
)

# 

,user_id,event_type,ip_address,is_threat,country,account_age_days,u1hr_events,u12hr_events,u24hr_events
timestamp,,,,,,,,,
2026-04-09 19:06:41.072217,0,password_reset,39.10.96.188,0,0,0.0,0.0,0.0,0.0
2026-04-09 22:34:41.099105,0,file_download,183.176.206.92,0,0,0.0,1.0,1.0,1.0
2026-04-10 00:12:41.372096,0,email_click,81.167.201.129,0,0,0.0,1.0,2.0,2.0
2026-04-10 07:15:41.032578,0,file_download,188.199.224.205,0,0,0.0,1.0,3.0,3.0
2026-04-10 23:15:41.215272,0,email_click,33.196.189.113,0,0,0.0,1.0,3.0,4.0
...,...,...,...,...,...,...,...,...,...
2026-05-05 14:37:41.079960,500,login,66.45.57.92,0,USA,118.0,1.0,2.0,3.0
2026-05-05 22:37:41.366141,500,password_reset,69.163.139.75,0,USA,118.0,1.0,2.0,3.0
2026-05-07 04:07:41.230185,500,password_reset,180.16.246.9,0,USA,118.0,1.0,3.0,4.0
